# 智慧垃圾分類 — YOLO11s 訓練 Notebook

**執行前必做：Runtime → Change runtime type → T4 GPU**

## 執行順序
1. Cell 1：確認 GPU
2. Cell 2：安裝套件
3. Cell 3：Clone 專案
4. Cell 4：下載 TrashNet 資料集（需 Kaggle API Token）
5. Cell 5：TrashNet 格式轉換
6. Cell 6：下載 TACO + 合併（約 20–40 分鐘，建議執行）
7. Cell 7：訓練
8. Cell 8：下載 best.pt

In [ ]:
# ── Cell 1：確認 GPU ───────────────────────────────────────────────────────────
import torch
print('GPU available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('Device:', torch.cuda.get_device_name(0))
else:
    raise RuntimeError('❌ 沒有 GPU，請先到 Runtime → Change runtime type → T4 GPU')

In [ ]:
# ── Cell 2：安裝套件 ───────────────────────────────────────────────────────────
!pip install ultralytics kaggle requests Pillow -q
print('套件安裝完成')

In [ ]:
# ── Cell 3：Clone 專案 ─────────────────────────────────────────────────────────
import os

REPO = 'AI-course'
if os.path.exists(REPO):
    print('Repo 已存在，執行 git pull...')
    !cd {REPO} && git pull
else:
    !git clone https://github.com/Saibusu/AI-course.git

%cd /content/AI-course
print('工作目錄：', os.getcwd())

In [ ]:
# ── Cell 4：下載 TrashNet（Kaggle API Token）───────────────────────────────────
# 取得方式：kaggle.com/settings → API → 複製 Token（格式：KGAT_...）
import os

os.environ['KAGGLE_TOKEN'] = 'YOUR_KAGGLE_API_TOKEN'  # ← 貼上你的 Token，勿上傳 GitHub

!kaggle datasets download -d asdasdasasdas/garbage-classification -p data/
!unzip -q data/garbage-classification.zip -d data/TrashNet

# 確認結構
for root, dirs, files in os.walk('data/TrashNet'):
    level = root.replace('data/TrashNet', '').count(os.sep)
    if level == 2:
        print(f'{os.path.basename(root)}/: {len(files)} files')
    if level > 2:
        break

In [ ]:
# ── Cell 5：TrashNet → YOLO 6-class 格式轉換 ──────────────────────────────────
# 已知：Kaggle 資料集實際路徑為兩層子目錄
# data/TrashNet/garbage classification/Garbage classification/{glass,metal,...}/
import os

TRASHNET_DIR = 'data/TrashNet/garbage classification/Garbage classification'

if not os.path.exists(TRASHNET_DIR):
    # 嘗試大寫版本
    TRASHNET_DIR = 'data/TrashNet/Garbage classification/Garbage classification'

print('TrashNet 來源路徑:', TRASHNET_DIR)
print('子目錄：', os.listdir(TRASHNET_DIR))

!python data/prepare_trashnet.py \
    --trashnet-dir "{TRASHNET_DIR}" \
    --output-dir   data/trashnet_yolo

for split in ['train', 'val', 'test']:
    imgs = len(list(os.scandir(f'data/trashnet_yolo/{split}/images')))
    print(f'  {split}: {imgs} images')

In [ ]:
# ── Cell 6：下載 TACO + 合併三層資料集 ────────────────────────────────────────
# ADR-002 Layer 1（主力）：含鋁箔包（Drink carton）標註
# 預計下載時間：20–40 分鐘（Flickr 圖片）
import os

# Step 1：Clone TACO repo（若已存在則跳過）
if not os.path.exists('data/TACO_repo'):
    !git clone https://github.com/pedropro/TACO.git data/TACO_repo
else:
    print('TACO_repo 已存在，跳過 clone')

# Step 2：確認 annotations.json 存在
ann_path = 'data/TACO_repo/data/annotations.json'
print('annotations.json 存在：', os.path.exists(ann_path))
print('TACO data/ 內容：', os.listdir('data/TACO_repo/data') if os.path.exists('data/TACO_repo/data') else 'NOT FOUND')

In [ ]:
# ── Cell 6b：下載 TACO 圖片（執行前確認 Cell 6 annotations.json 存在）─────────
%cd /content/AI-course/data/TACO_repo
!python download.py --dataset_path data/annotations.json
%cd /content/AI-course

# 確認下載後的結構
import os
data_contents = os.listdir('data/TACO_repo/data')
print('下載後 data/ 內容（前 10 項）：', data_contents[:10])
img_count = sum(1 for f in os.walk('data/TACO_repo/data') for ff in f[2] if ff.endswith(('.jpg','.JPG','.jpeg')))
print('下載圖片總數：', img_count)

In [ ]:
# ── Cell 6c：TACO 轉換 + 合併 ─────────────────────────────────────────────────
import os

!python data/prepare_taco.py \
    --taco-dir   data/TACO_repo/data \
    --output-dir data/taco_yolo

!python data/merge_datasets.py \
    --taco     data/taco_yolo \
    --trashnet data/trashnet_yolo \
    --output   data/merged

print('\n合併後資料集：')
for split in ['train', 'val', 'test']:
    imgs = len(list(os.scandir(f'data/merged/{split}/images')))
    print(f'  {split}: {imgs} images')

In [ ]:
# ── Cell 7：訓練 YOLO11s ──────────────────────────────────────────────────────
import os
from ultralytics import YOLO

# 如果 Cell 6 成功，用合併資料集；否則只用 TrashNet
if os.path.exists('data/merged/data.yaml'):
    DATA_YAML = 'data/merged/data.yaml'
    print('使用合併資料集（TACO + TrashNet）')
else:
    DATA_YAML = 'data/trashnet_yolo/data.yaml'
    print('⚠️ 僅使用 TrashNet（Class 4 鋁箔包 = 0 張）')

model = YOLO('yolo11s.pt')

results = model.train(
    data=DATA_YAML,
    epochs=50,
    imgsz=416,
    batch=16,
    device=0,
    project='runs/train',
    name='waste_sorter',
    exist_ok=True,
    patience=15,
    lr0=1e-3,
    lrf=1e-2,
    mosaic=1.0,
    fliplr=0.5,
    degrees=15.0,
    translate=0.1,
    scale=0.3,
)

mAP = results.results_dict.get('metrics/mAP50(B)', 'N/A')
print(f'\n訓練完成！mAP@50 = {mAP}')
print('模型路徑：runs/train/waste_sorter/weights/best.pt')

In [ ]:
# ── Cell 8：下載 best.pt ──────────────────────────────────────────────────────
import shutil, os
from google.colab import files

src = 'runs/train/waste_sorter/weights/best.pt'
shutil.copy(src, 'best.pt')
size_mb = os.path.getsize('best.pt') / 1e6
print(f'Model size: {size_mb:.1f} MB')

files.download('best.pt')
print('\n✅ 下載完成。接著在筆電執行：')
print('scp best.pt jetson@<JETSON_IP>:~/AI-course/models/')